In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [12]:
load_dotenv()

True

In [13]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-Coder-7B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.7,
)

In [14]:
model = ChatHuggingFace(llm = llm)

In [15]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [16]:
def chat_node(state: ChatState):
    messages=state['messages']
    response = model.invoke(messages)

    return {"messages": [response]}



In [17]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)


graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END )

chatbot = graph.compile(checkpointer=checkpointer)

In [18]:
initial_state = {
    'messages' : [HumanMessage(content='What is the capital of India')]
}

chatbot.invoke(initial_state)['messages'][-1].content

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [19]:
thread_id = 1

while True:
    user_message = input('Type here: ')
    print('User:', user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye']:
        break

    config = {'configurable': {'thread_id': thread_id}}

    response = chatbot.invoke({'messages': [HumanMessage(content = user_message)]}, config=config)

    print('AI:', response['messages'][-1].content)



User: hi
AI: Hello! How can I assist you today?
User: 10+29
AI: The sum of 10 and 29 is 39.
User: subtrat 9 from prev output
AI: Subtracting 9 from the previous output (39) gives us 30.
User: bye
